# Notebook 06 — Duplicate Marking and Base Quality Score Recalibration (BQSR)

**Module:** 09 — Genomics and NGS  
**Notebook:** 06 of 16  
**Time estimate:** 75 minutes

> Two GATK Best Practices steps with non-obvious rationale — you will be asked
> "why do you mark duplicates?" and "what is BQSR?" in almost every NGS interview.

---
## Step 1 — Motivation

PCR amplification introduces duplicate reads — copies of the same original molecule.
If not removed, a PCR artifact looks identical to a real variant at 50% allele
frequency. BQSR corrects systematic sequencing errors in quality scores, improving
the calibration of genotype likelihood models downstream.

---
## Step 2 — Intuition

**Duplicate marking:**  
If two reads have identical start positions, strands, and (for paired-end) identical
mate positions, they almost certainly came from the same original DNA molecule and
are PCR copies. Mark all but one copy with FLAG bit 0x400. Most variant callers
ignore marked duplicates.

Two types of duplicates:
- **PCR duplicates:** Same molecule amplified multiple times before sequencing
- **Optical duplicates:** Same cluster on the flowcell read twice (flowcell tile artifact)

**BQSR (Base Quality Score Recalibration):**  
The quality score Q on each base represents P(error) = 10^(-Q/10). But the machine's
score can be systematically off: cycles late in the read, certain k-mer contexts,
and specific sequencing chemistry can cause scores to be over- or under-estimated.
BQSR builds a covariate model from known-variant sites (dbSNP), then adjusts Q scores
to better match observed empirical error rates.

---
## Step 3 — Biological Background

**Why PCR duplicates are problematic for variant calling:**

Suppose a position has one T→A variant on one allele. In a diploid organism:
- Without duplicates: 50 reads REF, 50 reads ALT → VAF = 50%, correctly called het
- With duplicates if all ALT reads were duplicated 10×: 50 reads REF, 500 reads ALT
  → VAF > 90%, erroneously called hom ALT or flagged as artifact

**Why BQSR matters for sensitive variant calling:**  
Low-frequency somatic variants (VAF 1–5%) require that Q30 really means Q30,
not Q25 or Q35. Somatic callers (Mutect2) are especially sensitive to quality
score miscalibration.

**BQSR covariates used by GATK:**
1. Read group (sequencing run)
2. Reported quality score
3. Cycle position (position in read)
4. Dinucleotide context (preceding base)

---
## Step 4 — Mathematical Explanation

**Duplicate identification key:**  
Two paired reads are duplicates if and only if:
$$(\text{chrom}_1, \text{pos5'}_1, \text{strand}_1, \text{chrom}_2, \text{pos5'}_2, \text{strand}_2)$$
matches exactly, where pos5' is the 5' end position of each mate.

**BQSR recalibration model:**  
For each covariate combination $(\text{read\_group}, Q, \text{cycle}, \text{dinuc})$:
1. Count total observations $N$ and true errors $E$ at known non-variant sites
2. Empirical error rate: $p_{\text{emp}} = E / N$
3. Recalibrated Q: $Q_{\text{recal}} = -10 \log_{10}(p_{\text{emp}})$

**Calibration plot:** A perfectly calibrated run shows reported Q = empirical Q on a
45° diagonal. Over-estimation (machine too optimistic) → points above diagonal.
BQSR shifts each point onto the diagonal.

In [ ]:
# Step 6 — Python: Duplicate marking and BQSR calibration model
import numpy as np
from collections import defaultdict
from dataclasses import dataclass


@dataclass
class PairedRead:
    read_id: str
    chrom: str
    pos5_r1: int    # 5' position of read 1
    strand_r1: str  # '+' or '-'
    pos5_r2: int
    strand_r2: str
    mapq: int
    is_duplicate: bool = False


def mark_duplicates(reads: list[PairedRead]) -> tuple[list[PairedRead], dict]:
    """
    Mark PCR duplicates using 5' coordinate key.
    For each unique (chrom, pos5_r1, strand_r1, pos5_r2, strand_r2), keep
    the read with highest MAPQ; mark all others as duplicates.
    """
    # Group reads by duplicate key
    groups = defaultdict(list)
    for r in reads:
        key = (r.chrom, r.pos5_r1, r.strand_r1, r.pos5_r2, r.strand_r2)
        groups[key].append(r)

    n_marked = 0
    for key, group in groups.items():
        if len(group) == 1:
            continue
        # Sort by MAPQ descending, keep first, mark rest
        group.sort(key=lambda r: r.mapq, reverse=True)
        for r in group[1:]:
            r.is_duplicate = True
            n_marked += 1

    stats = {
        'total': len(reads),
        'duplicates': n_marked,
        'duplicate_rate': n_marked / len(reads) if reads else 0,
        'unique_molecules': len(groups),
    }
    return reads, stats


# Simulate paired reads with duplicates
def simulate_paired_reads(n_unique: int = 500, dup_rate: float = 0.15, seed: int = 42):
    rng = np.random.default_rng(seed)
    reads = []
    for i in range(n_unique):
        pos1 = int(rng.integers(1, 10000))
        strand1 = rng.choice(['+', '-'])
        pos2 = pos1 + int(rng.integers(150, 500))
        strand2 = '-' if strand1 == '+' else '+'
        mapq = int(rng.integers(30, 61))
        reads.append(PairedRead(f'r{i:04d}', 'chr1', pos1, strand1, pos2, strand2, mapq))
        # Add duplicates
        n_dups = int(rng.geometric(1 - dup_rate)) - 1
        for d in range(n_dups):
            dup_mapq = max(10, mapq + int(rng.integers(-5, 5)))
            reads.append(PairedRead(f'r{i:04d}_dup{d}', 'chr1', pos1, strand1, pos2, strand2, dup_mapq))
    return reads


reads = simulate_paired_reads(n_unique=500, dup_rate=0.18)
reads, dup_stats = mark_duplicates(reads)

print(f"Total reads:        {dup_stats['total']:,}")
print(f"Unique molecules:   {dup_stats['unique_molecules']:,}")
print(f"Marked duplicates:  {dup_stats['duplicates']:,} ({dup_stats['duplicate_rate']*100:.1f}%)")

In [ ]:
# BQSR: calibration model
def simulate_bqsr_data(n_bases: int = 100000, seed: int = 42) -> dict:
    """
    Simulate a quality score recalibration dataset.
    Returns {reported_q: (n_total, n_errors)} for each Q score bin.
    """
    rng = np.random.default_rng(seed)

    # Simulate reads with systematic overcalibration at high Q
    # Machine reports Q, but true error rate is worse than Q implies
    reported_q = rng.integers(10, 41, size=n_bases)

    # True error rate: machine overestimates quality
    # At Q30 (P_err=0.001), true error is P_err=0.003
    # At Q20 (P_err=0.01), true error is P_err=0.012 (slight overestimation)
    def true_error_rate(q: int) -> float:
        nominal = 10 ** (-q / 10)
        # Systematic bias: worse at high Q (late cycles)
        bias_factor = 1.5 + 0.05 * max(0, q - 20)
        return min(0.5, nominal * bias_factor)

    errors = np.array([rng.random() < true_error_rate(q) for q in reported_q])

    # Bin by reported Q
    calibration = {}
    for q in range(10, 41):
        mask = (reported_q == q)
        n = mask.sum()
        if n == 0:
            continue
        n_err = errors[mask].sum()
        calibration[q] = (int(n), int(n_err))

    return calibration


def apply_bqsr(calibration: dict) -> dict:
    """Compute recalibrated Q score for each reported Q."""
    recalibrated = {}
    for q, (n, n_err) in calibration.items():
        if n_err == 0:
            recalibrated[q] = q  # no errors observed: keep Q
        else:
            p_emp = n_err / n
            q_recal = -10 * np.log10(p_emp)
            recalibrated[q] = round(q_recal, 1)
    return recalibrated


calibration = simulate_bqsr_data()
recalibrated = apply_bqsr(calibration)

print("BQSR recalibration table (sample):")
print(f"{'Reported Q':>12} {'Empirical Q':>12} {'Delta':>8} {'N bases':>10}")
print('-' * 46)
for q in [15, 20, 25, 30, 35, 40]:
    if q in recalibrated:
        n, n_err = calibration[q]
        q_recal = recalibrated[q]
        print(f"{q:>12} {q_recal:>12.1f} {q_recal-q:>8.1f} {n:>10,}")

In [ ]:
# Step 7 — Visualization
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel A: Duplicate rate vs. library complexity
ax = axes[0, 0]
n_unique_values = np.logspace(5, 8, 100)
n_sequenced = 1e7  # 10M reads
# P(duplicate) ≈ 1 - (N_unique / N_sequenced) under occupancy model
dup_rates = 1 - n_unique_values / n_sequenced * (1 - np.exp(-n_sequenced / n_unique_values))
ax.semilogx(n_unique_values, dup_rates * 100, 'b-', lw=2)
ax.set_xlabel('Library complexity (unique molecules)')
ax.set_ylabel('Expected duplicate rate (%)')
ax.set_title('A. Duplicate rate vs. library complexity\n(10M reads sequenced)')
ax.axhline(20, color='orange', ls='--', label='20% threshold')
ax.legend(fontsize=8)

# Panel B: Effect of duplicate removal on allele frequency
ax2 = axes[0, 1]
np.random.seed(42)
# True VAF = 0.5 (heterozygous SNP)
# With high duplicate rate, sampled VAF is biased
dup_rates_sim = np.arange(0, 0.9, 0.05)
vaf_bias = []
for dr in dup_rates_sim:
    vafs = []
    for _ in range(200):
        # Simulate 100 reads, 50 unique REF, 50 unique ALT
        n_ref_unique = 50; n_alt_unique = 50
        # Add duplicates — assume duplicate rate is the same for both alleles
        n_reads = int(100 / (1 - dr)) if dr < 1 else 1000
        reads = np.random.binomial(n_reads, 0.5)
        vafs.append(reads / n_reads)
    vaf_bias.append(np.std(vafs))
ax2.plot(dup_rates_sim * 100, vaf_bias, 'r-', lw=2)
ax2.set_xlabel('Duplicate rate (%)')
ax2.set_ylabel('SD of estimated VAF')
ax2.set_title('B. VAF estimation variance increases\nwith duplicate rate')

# Panel C: BQSR calibration plot
ax3 = axes[1, 0]
q_vals = sorted(recalibrated.keys())
q_recal = [recalibrated[q] for q in q_vals]
ax3.plot([10, 40], [10, 40], 'k--', lw=1, label='Perfect calibration')
ax3.plot(q_vals, q_recal, 'ro-', lw=2, ms=4, label='Empirical (before BQSR)')
ax3.fill_between(q_vals, q_vals, q_recal, alpha=0.2, color='red',
                  label='Overcalibration (Q inflated)')
ax3.set_xlabel('Reported Q score')
ax3.set_ylabel('Empirical Q score')
ax3.set_title('C. BQSR calibration plot\n(points above diagonal = overcalibrated)')
ax3.legend(fontsize=8)
ax3.set_xlim(10, 40); ax3.set_ylim(10, 40)

# Panel D: GATK Best Practices pipeline steps
ax4 = axes[1, 1]
ax4.axis('off')
steps = [
    ('1. Align', 'BWA-MEM → SAM → sort → index'),
    ('2. Mark Dups', 'Picard MarkDuplicates (FLAG 0x400)'),
    ('3. BQSR Step 1', 'BaseRecalibrator: build covariate model'),
    ('4. BQSR Step 2', 'ApplyBQSR: write recalibrated BAM'),
    ('5. QC', 'samtools flagstat + coverage check'),
    ('6. Call', 'HaplotypeCaller → GVCF'),
]
colors4 = ['#E3F2FD', '#FFF3E0', '#E8F5E9', '#E8F5E9', '#F3E5F5', '#FCE4EC']
for i, (title, desc) in enumerate(steps):
    y = 0.88 - i * 0.14
    ax4.add_patch(plt.Rectangle((0.05, y - 0.06), 0.90, 0.11,
                                  facecolor=colors4[i], edgecolor='gray', lw=1.5))
    ax4.text(0.10, y, title, ha='left', va='center', fontsize=9, fontweight='bold')
    ax4.text(0.35, y, desc, ha='left', va='center', fontsize=8)
    if i < len(steps) - 1:
        ax4.annotate('', xy=(0.5, y - 0.06), xytext=(0.5, y - 0.03),
                     arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
ax4.set_title('D. GATK Best Practices: pre-processing pipeline', fontsize=10, fontweight='bold')
ax4.set_xlim(0, 1); ax4.set_ylim(0, 1)

plt.suptitle('Duplicate Marking and BQSR', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('duplicate_bqsr.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. A read pair has FLAG values 99 and 147. After duplicate marking, their duplicate
   copies get FLAG values 1123 and 1171. Verify this by adding 0x400 (1024) to the
   original flags and computing the result.
2. Describe the four covariates used in GATK's BQSR and explain why each might
   systematically affect quality score accuracy.
3. Under what circumstances would you skip duplicate marking? (Hint: think about
   amplicon sequencing.)

---
## Step 10 — Quiz

1. What is the difference between PCR and optical duplicates?
2. Why is BQSR performed on known-variant sites from dbSNP rather than all sites?
3. A BQSR calibration plot shows points below the diagonal (empirical Q < reported Q).
   Is the machine over- or under-estimating quality? What does BQSR do to correct this?
4. What FLAG bit is set on a duplicate read?
5. Name two applications where you would NOT want to remove duplicates.

---
## Step 12 — Reflection

> *[In one paragraph: explain BQSR to someone who knows what a Phred score is but
> has never heard of recalibration. Why does the sequencer get the scores slightly
> wrong, and how does BQSR fix this?]*

---
*Next: `07_variant_calling_pileup_and_genotyping.ipynb`*